# Graph RAG Tutorial Part 2: Embedding and Retrieval

This notebook builds on GraphRAG01 and demonstrates:
1. Setting up ChromaDB for vector storage
2. Embedding graph information using sentence transformers
3. Building a retrieval system for graph queries
4. Creating a GraphRAG class for structured retrieval

## Prerequisites

```bash
pip install chromadb sentence-transformers plotly
```

## 1. Setup and Imports

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go

# For RAG components
try:
    import chromadb
    from sentence_transformers import SentenceTransformer
    RAG_AVAILABLE = True
    print("ChromaDB and SentenceTransformers available")
except ImportError as e:
    RAG_AVAILABLE = False
    print(f"Warning: RAG components not available: {e}")
    print("Install with: pip install chromadb sentence-transformers")

## 2. Create a Building Model

We'll create a more complex building model with multiple rooms to demonstrate GraphRAG capabilities.

In [ ]:
# Create a 3x2 grid of rooms simulating a small office
room_definitions = [
    {"id": "c_1", "name": "Reception", "category": "public", "x": 0, "y": 0},
    {"id": "c_2", "name": "Conference Room", "category": "meeting", "x": 3, "y": 0},
    {"id": "c_3", "name": "Office 1", "category": "workspace", "x": 6, "y": 0},
    {"id": "c_4", "name": "Kitchen", "category": "amenity", "x": 0, "y": 3},
    {"id": "c_5", "name": "Break Room", "category": "amenity", "x": 3, "y": 3},
    {"id": "c_6", "name": "Office 2", "category": "workspace", "x": 6, "y": 3},
]

# Create cells (rooms)
rooms = []
for rd in room_definitions:
    room = tf.Cell.Box(rd["x"], rd["y"], 0, 3, 3, 3)  # 3x3x3 meter rooms
    rooms.append(room)

# Build CellComplex
building = tf.CellComplex.ByCells(rooms)

print(f"Building created:")
print(f"  Rooms: {building.NumCells()}")
print(f"  Total volume: {building.Volume():.1f} m³")

# Create connectivity graph
graph = tf.Graph.ByTopology(building)

print(f"\nConnectivity graph:")
print(f"  Vertices: {graph.Order()}")
print(f"  Edges: {graph.Size()}")
print(f"  Diameter: {graph.Diameter()}")

## 3. Convert Graph to Text Descriptions

For embedding, we need to convert graph structure into rich text descriptions.

In [ ]:
def graph_to_texts(graph, room_definitions, mantissa=2):
    """
    Convert graph to text descriptions with room metadata.
    
    Returns list of text descriptions suitable for embedding.
    """
    texts = []
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Map vertex positions to room definitions
    vertex_to_room = {}
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        # Find closest room definition
        for rd in room_definitions:
            # Room center is at (x+1.5, y+1.5, 1.5) for 3x3x3 rooms
            center_x = rd["x"] + 1.5
            center_y = rd["y"] + 1.5
            if abs(coords[0] - center_x) < 0.1 and abs(coords[1] - center_y) < 0.1:
                vertex_to_room[i] = rd
                break
        if i not in vertex_to_room:
            vertex_to_room[i] = {"id": f"v_{i}", "name": f"Unknown_{i}", "category": "unknown"}
    
    # Create vertex descriptions
    for i, v in enumerate(vertices):
        rd = vertex_to_room[i]
        coords = v.Coordinates()
        degree = graph.VertexDegree(v)
        
        # Get adjacent rooms
        adjacent = graph.AdjacentVertices(v)
        adj_names = []
        for adj_v in adjacent:
            adj_coords = adj_v.Coordinates()
            for j, orig_v in enumerate(vertices):
                orig_coords = orig_v.Coordinates()
                if (abs(adj_coords[0] - orig_coords[0]) < 0.01 and 
                    abs(adj_coords[1] - orig_coords[1]) < 0.01):
                    adj_names.append(vertex_to_room[j]["name"])
                    break
        
        text = f"Room: {rd['name']} (ID: {rd['id']}, Category: {rd['category']}). " \
               f"Located at coordinates ({coords[0]:.{mantissa}f}, {coords[1]:.{mantissa}f}, {coords[2]:.{mantissa}f}). " \
               f"Has {degree} connection(s) to: {', '.join(adj_names) if adj_names else 'none'}."
        texts.append(text)
    
    # Create edge descriptions
    for i, edge in enumerate(edges):
        edge_verts = edge.Vertices()
        if len(edge_verts) >= 2:
            # Find rooms for both endpoints
            room1_name = "Unknown"
            room2_name = "Unknown"
            
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            
            for j, v in enumerate(vertices):
                vc = v.Coordinates()
                if abs(p1[0] - vc[0]) < 0.01 and abs(p1[1] - vc[1]) < 0.01:
                    room1_name = vertex_to_room[j]["name"]
                if abs(p2[0] - vc[0]) < 0.01 and abs(p2[1] - vc[1]) < 0.01:
                    room2_name = vertex_to_room[j]["name"]
            
            text = f"Connection: {room1_name} is connected to {room2_name} (shared wall)."
            texts.append(text)
    
    return texts, vertex_to_room

# Generate texts
texts, vertex_to_room = graph_to_texts(graph, room_definitions)

print("Generated text descriptions:")
print("=" * 60)
for i, text in enumerate(texts):
    print(f"\n[{i}] {text}")

## 4. Set Up ChromaDB and Embeddings

We'll use ChromaDB to store vector embeddings and enable similarity search.

In [ ]:
if RAG_AVAILABLE:
    # Initialize embedding model
    # Using a small, fast model suitable for semantic similarity
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    print(f"Loaded embedding model: all-MiniLM-L6-v2")
    
    # Initialize ChromaDB (in-memory for this demo)
    chroma_client = chromadb.Client()
    
    # Create a collection for our building graph
    # Delete if exists (for re-running the notebook)
    try:
        chroma_client.delete_collection(name='building_graph')
    except:
        pass
    
    collection = chroma_client.create_collection(name='building_graph')
    print(f"Created ChromaDB collection: building_graph")
else:
    print("Skipping embedding setup - RAG components not available")

## 5. Embed Graph Information

In [ ]:
def embed_graph(texts, collection, embedder):
    """
    Embed graph texts and store in ChromaDB.
    """
    # Generate embeddings
    embeddings = embedder.encode(texts).tolist()
    
    # Store in collection
    collection.add(
        embeddings=embeddings,
        documents=texts,
        ids=[f"item_{i}" for i in range(len(texts))]
    )
    
    return len(texts)

if RAG_AVAILABLE:
    num_embedded = embed_graph(texts, collection, embedder)
    print(f"Embedded {num_embedded} text descriptions into ChromaDB")
else:
    print("Skipping embedding - RAG components not available")

## 6. Implement Retrieval Functions

In [ ]:
def retrieve_relevant_nodes(query, collection, embedder, top_k=3):
    """
    Retrieve the most relevant graph information for a query.
    
    Parameters:
    - query: Natural language question
    - collection: ChromaDB collection
    - embedder: SentenceTransformer model
    - top_k: Number of results to return
    
    Returns:
    - List of relevant document texts
    """
    # Embed the query
    query_embedding = embedder.encode(query).tolist()
    
    # Query ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    
    return results['documents'][0], results['distances'][0]

if RAG_AVAILABLE:
    # Test retrieval with sample queries
    test_queries = [
        "Which rooms are connected to the Kitchen?",
        "Where are the workspaces located?",
        "What is connected to the Reception?",
        "How many connections does Office 1 have?",
    ]
    
    print("Testing retrieval:")
    print("=" * 60)
    
    for query in test_queries:
        docs, distances = retrieve_relevant_nodes(query, collection, embedder, top_k=2)
        print(f"\nQuery: {query}")
        print(f"Top results:")
        for i, (doc, dist) in enumerate(zip(docs, distances)):
            print(f"  [{i+1}] (dist={dist:.3f}) {doc[:100]}...")
else:
    print("Skipping retrieval test - RAG components not available")

## 7. Build a GraphRAG Class

Let's create a reusable class that encapsulates the GraphRAG functionality.

In [ ]:
class GraphRAG:
    """
    A Graph-based Retrieval-Augmented Generation system using topologic_fast.
    
    This class stores knowledge in a graph structure and provides retrieval
    capabilities using semantic similarity.
    """
    
    def __init__(self, embedding_model='all-MiniLM-L6-v2'):
        """
        Initialize the GraphRAG system.
        
        Parameters:
        - embedding_model: Name of the sentence transformer model to use
        """
        if not RAG_AVAILABLE:
            raise ImportError("chromadb and sentence-transformers are required")
        
        self.embedder = SentenceTransformer(embedding_model)
        self.chroma_client = chromadb.Client()
        self.collection = None
        self.graph = None
        self.texts = []
        self.metadata = {}
    
    def load_graph(self, graph, room_definitions=None, collection_name='graph_rag'):
        """
        Load a topologic_fast graph into the RAG system.
        
        Parameters:
        - graph: tf.Graph object
        - room_definitions: List of dicts with room metadata
        - collection_name: Name for the ChromaDB collection
        """
        self.graph = graph
        
        # Create or replace collection
        try:
            self.chroma_client.delete_collection(name=collection_name)
        except:
            pass
        
        self.collection = self.chroma_client.create_collection(name=collection_name)
        
        # Convert graph to texts
        if room_definitions:
            self.texts, self.metadata = graph_to_texts(graph, room_definitions)
        else:
            self.texts = self._basic_graph_to_texts(graph)
        
        # Embed and store
        embeddings = self.embedder.encode(self.texts).tolist()
        self.collection.add(
            embeddings=embeddings,
            documents=self.texts,
            ids=[f"item_{i}" for i in range(len(self.texts))]
        )
        
        return len(self.texts)
    
    def _basic_graph_to_texts(self, graph):
        """Basic conversion without metadata."""
        texts = []
        vertices = graph.Vertices()
        
        for i, v in enumerate(vertices):
            coords = v.Coordinates()
            degree = graph.VertexDegree(v)
            text = f"Vertex {i} at ({coords[0]:.2f}, {coords[1]:.2f}, {coords[2]:.2f}) with {degree} connections."
            texts.append(text)
        
        return texts
    
    def retrieve(self, query, top_k=3):
        """
        Retrieve relevant graph information for a query.
        
        Parameters:
        - query: Natural language question
        - top_k: Number of results to return
        
        Returns:
        - List of relevant document texts with distances
        """
        if self.collection is None:
            raise ValueError("No graph loaded. Call load_graph() first.")
        
        query_embedding = self.embedder.encode(query).tolist()
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )
        
        return list(zip(results['documents'][0], results['distances'][0]))
    
    def get_context(self, query, top_k=3):
        """
        Get formatted context string for LLM prompts.
        
        Parameters:
        - query: Natural language question
        - top_k: Number of results to include
        
        Returns:
        - Formatted context string
        """
        results = self.retrieve(query, top_k)
        context_parts = [doc for doc, _ in results]
        return "\n".join(context_parts)

print("GraphRAG class defined successfully")

## 8. Test the GraphRAG Class

In [ ]:
if RAG_AVAILABLE:
    # Create GraphRAG instance
    rag = GraphRAG()
    
    # Load our building graph
    num_items = rag.load_graph(graph, room_definitions, 'office_building')
    print(f"Loaded {num_items} items into GraphRAG")
    
    # Test queries
    print("\n" + "=" * 60)
    print("Testing GraphRAG queries:")
    print("=" * 60)
    
    queries = [
        "What rooms are connected to the Kitchen?",
        "Where can I find workspaces?",
        "Which room has the most connections?",
        "Is the Conference Room connected to Reception?"
    ]
    
    for query in queries:
        print(f"\nQ: {query}")
        context = rag.get_context(query, top_k=2)
        print(f"Context retrieved:")
        for line in context.split('\n'):
            print(f"  > {line}")
else:
    print("Skipping GraphRAG test - RAG components not available")

## 9. Visualize the Building Graph

In [ ]:
# Create a 2D floor plan visualization
fig = go.Figure()

# Color scheme by category
category_colors = {
    'public': '#87CEEB',
    'meeting': '#FFD700',
    'workspace': '#90EE90',
    'amenity': '#FFB6C1'
}

# Draw rooms
for rd in room_definitions:
    x0, y0 = rd["x"], rd["y"]
    x = [x0, x0+3, x0+3, x0, x0]
    y = [y0, y0, y0+3, y0+3, y0]
    color = category_colors.get(rd["category"], '#CCCCCC')
    
    fig.add_trace(go.Scatter(
        x=x, y=y,
        fill='toself',
        fillcolor=color,
        line=dict(color='black', width=2),
        name=f"{rd['name']} ({rd['category']})",
        hoverinfo='name'
    ))
    
    # Add room label
    fig.add_annotation(
        x=x0+1.5, y=y0+1.5,
        text=rd["name"],
        showarrow=False,
        font=dict(size=10)
    )

# Draw graph edges
graph_edges = graph.Edges()
for edge in graph_edges:
    edge_verts = edge.Vertices()
    if len(edge_verts) >= 2:
        p1 = edge_verts[0].Coordinates()
        p2 = edge_verts[1].Coordinates()
        fig.add_trace(go.Scatter(
            x=[p1[0], p2[0]],
            y=[p1[1], p2[1]],
            mode='lines',
            line=dict(color='red', width=4),
            showlegend=False,
            hoverinfo='skip'
        ))

# Draw graph vertices
graph_verts = graph.Vertices()
coords = [v.Coordinates() for v in graph_verts]
x = [c[0] for c in coords]
y = [c[1] for c in coords]

fig.add_trace(go.Scatter(
    x=x, y=y,
    mode='markers',
    marker=dict(size=15, color='red', line=dict(color='darkred', width=2)),
    name='Graph Nodes',
    hovertext=[rd["name"] for rd in room_definitions],
    hoverinfo='text'
))

fig.update_layout(
    title='Office Floor Plan with Connectivity Graph',
    xaxis=dict(title='X (m)', scaleanchor='y', scaleratio=1, range=[-1, 10]),
    yaxis=dict(title='Y (m)', range=[-1, 7]),
    width=900,
    height=500,
    showlegend=True,
    legend=dict(x=1.02, y=1)
)

fig.show()

## Summary

In this notebook, we covered:

1. **Creating a building model** with multiple rooms using topologic_fast
2. **Converting graphs to text** with rich metadata for embedding
3. **Setting up ChromaDB** for vector storage
4. **Embedding graph information** using sentence transformers
5. **Implementing retrieval** based on semantic similarity
6. **Building a GraphRAG class** for reusable RAG functionality

### Key Takeaways

- **Text representation matters** - Rich descriptions enable better semantic search
- **Metadata enhances retrieval** - Room categories, names, and connections provide context
- **Vector similarity works well** - Even simple models can find relevant graph nodes

### Next Steps

In GraphRAG03, we will:
- Integrate with LLMs for response generation
- Build an interactive graph expansion system
- Implement structural graph matching